In [ ]:
# Firedrake coupled Saarelma–Connor solver tests
#
# Run from `tests/` (so `ROOT` points at the repo). Requires Firedrake +
# parent-class deps (OpenFUSIONToolkit, eqdsk/p-file inputs).

import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

INPUT_DIR = ROOT / "src" / "inputs" / "PT_Hmode"
MHD_FP = INPUT_DIR / "g152960.03500"
KPROF_FP = INPUT_DIR / "p152960.03500"

# `solver-firedrake.py` has a hyphen — load by file path.
_fd_spec = importlib.util.spec_from_file_location(
    "solver_firedrake",
    ROOT / "src" / "solver-firedrake.py",
)
_fd_mod = importlib.util.module_from_spec(_fd_spec)
_fd_spec.loader.exec_module(_fd_mod)
saarelma_connor_firedrake = _fd_mod.saarelma_connor_firedrake

# Shared solve kwargs (override per cell as needed).
SOLVE_KW = dict(
    x_res=200,
    mesh_n=400,
    fe_degree=2,
    use_picard=True,
    picard_tol=1e-4,
    max_picard=40,
    relax=0.5,
    ne_inner_bc="neumann",   # Saarelma A7 default; see dirichlet comparison below
    linear_solver="lu",      # or "gamg" for GMRES + algebraic multigrid on J
    verbose=True,
)


In [ ]:
model = saarelma_connor_firedrake(
    P_tot_e=5e6,
    alpha_crit=1.0,
    C_KBM=0.1,
    De_chie_etg=0.5,
    nFC_x0=1e15,          # m^-3 at separatrix
    mhd_fp=str(MHD_FP),
    kprof_fp=str(KPROF_FP),
    # nFC_threshold=0.01, nCX_threshold=0.01,  # adaptive x_inner (default)
    # psi_N_inner_boundary=0.97,               # fixed inner boundary instead
)

# Pre-solve grid setup (no Firedrake yet): locates x_inner from neutral thresholds.
model.form_factor(type="FC")
model.form_factor(type="cx")
model.setup_solver_grids(res=SOLVE_KW["x_res"])
model.find_inner_boundary()

print(f"psi_N inner = {model.psi_N_inner_boundary:.4f}")
print(f"x_inner     = {model.x_inner:.4e} m")
print(f"ne_x0       = {model.ne_x0:.3e} m^-3")
print(f"nFC_x0      = {model.nFC_x0:.3e} m^-3")
print(f"nCX_x0      = {model.nCX_x0:.3e} m^-3")

In [ ]:
# --- Baseline: Neumann n_e at x_inner, linear initial guess, direct LU ---
x_sol, ne_sol, nFC_sol, nCX_sol = model.solve_firedrake(
  **SOLVE_KW,
  initial_guess="linear",
)

# Converged profiles (x bottom, psi_N top on each panel)
model._plot_profiles(
    x_sol, ne_sol, nFC_sol, nCX_sol,
    title="Converged (linear IC, Neumann @ x_inner, LU)",
)

In [ ]:
# --- Tanh pedestal initial guess (plots IC when verbose=True, then solution) ---
# Re-use the same model instance; solve_firedrake re-runs setup each call.
x_sol_tanh, ne_tanh, nFC_tanh, nCX_tanh = model.solve_firedrake(
    **SOLVE_KW,
    initial_guess="tanh",
    tanh_width=None,    # default 0.1 * |x_inner|
    tanh_center=None,   # default -tanh_width
)

model._plot_profiles(
    x_sol_tanh, ne_tanh, nFC_tanh, nCX_tanh,
    title="Converged (tanh IC, Neumann @ x_inner, LU)",
)

In [ ]:
# --- p-file-based initial guess (n_e from p-file, FC exponential from separatrix) ---
x_sol_pf, ne_pf, nFC_pf, nCX_pf = model.solve_firedrake(
    **SOLVE_KW,
    initial_guess="pfile",
)

model._plot_profiles(
    x_sol_pf, ne_pf, nFC_pf, nCX_pf,
    title="Converged (pfile IC, Neumann @ x_inner, LU)",
)

In [ ]:
# --- Compare Neumann vs Dirichlet n_e at x_inner (same linear IC) ---
results_bc = {}
for bc in ("neumann", "dirichlet"):
    kw = {**SOLVE_KW, "ne_inner_bc": bc, "initial_guess": "linear", "verbose": False}
    results_bc[bc] = model.solve_firedrake(**kw)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fig.suptitle(r"$n_e$ inner BC: Neumann vs Dirichlet (linear IC)")

labels = {"neumann": "Neumann", "dirichlet": "Dirichlet"}
colors = {"neumann": "tab:blue", "dirichlet": "tab:orange"}
fields = [
    (lambda r: r[1] / 1e19, r"$n_e$ ($10^{19}$ m$^{-3}$)"),
    (lambda r: r[2] / 1e15, r"$\langle n_{FC}\rangle$ ($10^{15}$ m$^{-3}$)"),
    (lambda r: r[3] / 1e15, r"$\langle n_{CX}\rangle$ ($10^{15}$ m$^{-3}$)"),
]

for ax, (fn, ylab) in zip(axes, fields):
    for bc, (x, ne, nfc, ncx) in results_bc.items():
        ax.plot(x, fn((x, ne, nfc, ncx)), lw=2, color=colors[bc], label=labels[bc])
    ax.axvline(0, color="k", ls="--", lw=0.8)
    ax.set_xlabel(r"$x$ (m)")
    ax.set_ylabel(ylab)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.show()

In [ ]:
# --- Optional: GMRES + algebraic multigrid (GAMG) on each Newton Jacobian ---
# For this 1D problem LU is usually faster; use gamg when mesh_n / fe_degree is large.
x_sol_mg, ne_mg, nFC_mg, nCX_mg = model.solve_firedrake(
    **{**SOLVE_KW, "linear_solver": "gamg", "ksp_rtol": 1e-8, "ksp_max_it": 200},
    initial_guess="linear",
)

model._plot_profiles(
    x_sol_mg, ne_mg, nFC_mg, nCX_mg,
    title="Converged (linear IC, Neumann, GAMG)",
)

In [ ]:
# --- Optional: full Newton (no Picard outer loop) — less robust, good for debugging ---
# x_sol_newton, ne_n, nFC_n, nCX_n = model.solve_firedrake(
#     **{**SOLVE_KW, "use_picard": False},
#     initial_guess="pfile",
# )
# model._plot_profiles(x_sol_newton, ne_n, nFC_n, nCX_n, title="Full Newton, pfile IC")